# 09 — UV Abundances: C/O and N/O from Rest-Frame UV Lines

**Purpose:** Derive C/O and UV-based N/O from UV semi-forbidden and
resonance lines in a dust-corrected composite spectrum.  Demonstrates
the new UV abundance support in `jwspecabund`.

### Lines used

| Ion | Lines | Abundance |
|-----|-------|-----------|
| C2+ | CIII] 1907 + 1909 | C++/H+ |
| C3+ | CIV 1548 + 1551 | C+++/H+ |
| N2+ | NIII] 1749 + 1752 | N++/H+ |
| N3+ | NIV] 1483 + 1486 | N+++/H+ |
| N4+ | NV 1239 + 1243 | N4+/H+ (manual emissivity) |
| O2+ | [OIII] 4959 + 5007 | O++/H+ |

### Methodology

1. Fit UV + optical lines on the rest-frame stacked spectrum
2. Dust correction from Balmer decrement (Ha/Hb)
3. Electron temperature from [OIII] 4363/(4959+5007)
4. Electron density from CIII] 1907/1909
5. Ionic abundances via PyNEB (+ manual NV emissivity)
6. C/O = (C2+ + C3+) / O2+ (Jones+2023)
7. N/O = (N2+ + N3+ + N4+) / O2+
8. MC error propagation with pre-computed emissivity grids

**Data:** `stack_all_Muv19_21_DustCorrected.npz` — rest-frame composite
spectrum (z = 0, 1000–7500 Å, 129 galaxies).

**Date:** 2026-02-23

In [ ]:
import jwspecfit
from jwspecfit.lines import REST_LINES_A
from jwspecfit.resolution import R_from_pixels
from jwspecabund.dust import salim_attenuation, compute_Av_from_balmer
from jwspecabund.forward import hbeta_emissivity_aller84, _nv_emissivity
import matplotlib.pyplot as plt
import numpy as np
import pyneb as pn

print(f"jwspecfit  v{jwspecfit.__version__}")
print(f"pyneb     v{pn.__version__}")

## 1. Load the stacked spectrum

In [ ]:
spec = jwspecfit.read_npz("../../data/stack_all_Muv19_21_DustCorrected.npz", z=0.0)

# Estimate R from pixel spacing
R_func = R_from_pixels(spec.wave_um)

v = spec.mask_valid()
print(f"Pixels:     {spec.n_pix} ({v.sum()} valid)")
print(f"Wave range: {spec.wave_um[v].min()*1e4:.0f} – {spec.wave_um[v].max()*1e4:.0f} Å")
print(f"R at 1500 Å: {R_func(0.15):.0f}")
print(f"R at 5000 Å: {R_func(0.50):.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.step(spec.wave_um[v] * 1e4, spec.flux_ujy[v], where="mid", lw=0.5, color="0.3")
ax.fill_between(
    spec.wave_um[v] * 1e4,
    (spec.flux_ujy - spec.err_ujy)[v],
    (spec.flux_ujy + spec.err_ujy)[v],
    step="mid", alpha=0.15, color="0.5",
)
ax.set_xlabel(r"Rest wavelength [Å]")
ax.set_ylabel(r"Flux [arb.]")
ax.set_title("Dust-corrected composite spectrum (129 galaxies)")
ax.set_ylim(-2, 30)
plt.tight_layout()

## 2. Fit UV lines (~1200–2000 Å)

In [ ]:
uv_lines = [
    "NV_1", "NV_2",
    "NIV_1483", "NIV_1486",
    "CIV_1", "CIV_2",
    "HEII_1640",
    "OIII_1661", "OIII_1666",
    "NIII_1749", "NIII_1752",
    "SiIII_1", "SiIII_2",
    "CIII]_1907", "CIII]",
]

# Rest-frame spectrum at z=0, so observed = rest
result_uv = jwspecfit.fit_lines(
    spec, z=0.0, R=R_func,
    lines=uv_lines,
    wave_range_A=(1200, 2000),
    deg=3, n_boot=1000,
)

print(f"Lines fitted: {len(result_uv.lines)}")
print(f"chi2/dof:     {result_uv.chi2:.2f}")
print()
print(f"{'Line':<18s} {'Flux':>12s} {'Err':>12s} {'SNR':>8s}")
print("-" * 52)
for name, lr in result_uv.lines.items():
    print(f"{name:<18s} {lr.flux:12.3e} {lr.flux_err:12.3e} {lr.snr:8.1f}")

In [ ]:
fig = jwspecfit.plot_fit(result_uv)
fig.suptitle("UV line fit (1200–2000 Å)", y=1.02)
plt.show()

## 3. Fit optical lines (~4800–6800 Å)

In [ ]:
optical_lines = [
    "HBETA", "OIII_4363", "OIII_4959", "OIII_5007",
    "Ha", "NII_6585", "SII_6718", "SII_6732",
]

result_opt = jwspecfit.fit_lines(
    spec, z=0.0, R=R_func,
    lines=optical_lines,
    wave_range_A=(4300, 6800),
    deg=3, n_boot=1000,
)

print(f"Lines fitted: {len(result_opt.lines)}")
print(f"chi2/dof:     {result_opt.chi2:.2f}")
print()
print(f"{'Line':<18s} {'Flux':>12s} {'Err':>12s} {'SNR':>8s}")
print("-" * 52)
for name, lr in result_opt.lines.items():
    print(f"{name:<18s} {lr.flux:12.3e} {lr.flux_err:12.3e} {lr.snr:8.1f}")

In [ ]:
fig = jwspecfit.plot_fit(result_opt)
fig.suptitle("Optical line fit (4300–6800 Å)", y=1.02)
plt.show()

## 4. Dust correction

Derive A_V from the Ha/Hb Balmer decrement.  The spectrum is labelled
as dust-corrected in the filename but we verify here.

In [ ]:
def get_flux(result, name):
    lr = result.lines.get(name)
    if lr is None:
        return np.nan, np.nan
    return lr.flux, lr.flux_err

f_Ha, e_Ha = get_flux(result_opt, "Ha")
f_Hb, e_Hb = get_flux(result_opt, "HBETA")

print(f"Ha/Hb = {f_Ha/f_Hb:.3f}  (Case B = 2.86)")

Av_val, Av_err = compute_Av_from_balmer(
    f_Ha, f_Hb, e_Ha, e_Hb,
    law="salim", intrinsic_ratio=2.86,
    wave_num_A=REST_LINES_A["Ha"],
    wave_den_A=REST_LINES_A["HBETA"],
)
Av = max(Av_val, 0.0)
print(f"A_V = {Av:.3f} +/- {Av_err:.3f}")

# Dust correction helper
def dust_correct(flux, flux_err, rest_wave_A, Av):
    if Av <= 0 or not np.isfinite(Av):
        return flux, flux_err
    A_lam = salim_attenuation(np.array([rest_wave_A]), Av)[0]
    factor = 10.0 ** (0.4 * A_lam)
    return flux * factor, flux_err * factor

## 5. Electron temperature and density

In [ ]:
O3 = pn.Atom('O', 3)
C3 = pn.Atom('C', 3)
N3_atom = pn.Atom('N', 3)
N4_atom = pn.Atom('N', 4)
C4 = pn.Atom('C', 4)
H1 = pn.RecAtom('H', 1)

# Dust-corrected optical fluxes
f_4363, e_4363 = get_flux(result_opt, "OIII_4363")
f_4959, e_4959 = get_flux(result_opt, "OIII_4959")
f_5007, e_5007 = get_flux(result_opt, "OIII_5007")

f_4363_c, _ = dust_correct(f_4363, e_4363, 4364.44, Av)
f_4959_c, _ = dust_correct(f_4959, e_4959, 4961.68, Av)
f_5007_c, _ = dust_correct(f_5007, e_5007, 5009.64, Av)

ratio_oiii = f_4363_c / (f_4959_c + f_5007_c)
print(f"[OIII] 4363 SNR = {f_4363/e_4363:.1f}")
print(f"[OIII] 4363/(4959+5007) = {ratio_oiii:.4f}")

# Te from [OIII]
ne_init = 1000.0  # initial guess
Te_high = O3.getTemDen(
    ratio_oiii, den=ne_init,
    to_eval="L(4363) / (L(4959) + L(5007))",
    start_x=3.5, end_x=5.0, log=True,
)
print(f"Te(O2+) = {Te_high:.0f} K")

# ne from CIII] 1907/1909
f_c3_07, e_c3_07 = get_flux(result_uv, "CIII]_1907")
f_c3_09, e_c3_09 = get_flux(result_uv, "CIII]")

ne = ne_init
if f_c3_07 > 0 and f_c3_09 > 0:
    f_c3_07_c, _ = dust_correct(f_c3_07, e_c3_07, 1906.68, Av)
    f_c3_09_c, _ = dust_correct(f_c3_09, e_c3_09, 1908.73, Av)
    ratio_ciii = f_c3_07_c / f_c3_09_c
    ne_ciii = C3.getTemDen(ratio_ciii, tem=Te_high, wave1=1907, wave2=1909)
    if np.isfinite(ne_ciii) and ne_ciii > 0:
        ne = float(ne_ciii)
        print(f"CIII] 1907/1909 = {ratio_ciii:.3f}  ->  ne = {ne:.0f} cm^-3")
    else:
        print(f"CIII] ratio = {ratio_ciii:.3f} (outside physical range, using ne={ne:.0f})")
else:
    print(f"CIII] not detected, using ne = {ne:.0f} cm^-3")

# Recompute Te with refined ne
Te_high = O3.getTemDen(
    ratio_oiii, den=ne,
    to_eval="L(4363) / (L(4959) + L(5007))",
    start_x=3.5, end_x=5.0, log=True,
)
print(f"\nFinal Te(O2+) = {Te_high:.0f} K  (ne = {ne:.0f} cm^-3)")

## 6. Ionic abundances

All UV ionic abundances use Te(O2+) — appropriate for the high-ionisation zone.

In [ ]:
def get_j_Hb(Te, ne):
    """Hbeta emissivity with Aller 1984 fallback for Te > 30,000 K."""
    try:
        val = H1.getEmissivity(Te, ne, wave=4861)
        if np.isfinite(val):
            return val
    except Exception:
        pass
    return hbeta_emissivity_aller84(Te)


f_Hb_c, e_Hb_c = dust_correct(f_Hb, e_Hb, 4864.04, Av)
j_Hb = get_j_Hb(Te_high, ne)

print(f"{'Ion':<12s} {'Lines':<20s} {'Flux':>12s} {'X/H+':>12s} {'12+log':>10s}")
print("-" * 68)

# O2+/H+ from [OIII] 5007
j_5007 = O3.getEmissivity(Te_high, ne, wave=5007)
Opp = (f_5007_c / f_Hb_c) * (j_Hb / j_5007)
print(f"{'O2+':<12s} {'[OIII] 5007':<20s} {f_5007_c:12.3e} {Opp:12.3e} {12+np.log10(Opp):10.3f}")

# C2+/H+ from CIII] 1907+1909
f_ciii = 0.0
for name in ["CIII]_1907", "CIII]"]:
    f, e = get_flux(result_uv, name)
    if np.isfinite(f) and f > 0:
        f_ciii += f
Cpp = np.nan
if f_ciii > 0:
    f_ciii_c, _ = dust_correct(f_ciii, 0, 1908.0, Av)
    eps = C3.getEmissivity(Te_high, ne, wave=1907) + C3.getEmissivity(Te_high, ne, wave=1909)
    Cpp = (f_ciii_c / f_Hb_c) * (j_Hb / eps)
    print(f"{'C2+':<12s} {'CIII] 1907+1909':<20s} {f_ciii_c:12.3e} {Cpp:12.3e} {12+np.log10(Cpp):10.3f}")

# C3+/H+ from CIV 1548+1551
f_civ = 0.0
for name in ["CIV_1", "CIV_2"]:
    f, e = get_flux(result_uv, name)
    if np.isfinite(f) and f > 0:
        f_civ += f
Cppp = np.nan
if f_civ > 0:
    f_civ_c, _ = dust_correct(f_civ, 0, 1549.0, Av)
    eps_civ = C4.getEmissivity(Te_high, ne, wave=1548) + C4.getEmissivity(Te_high, ne, wave=1551)
    Cppp = (f_civ_c / f_Hb_c) * (j_Hb / eps_civ)
    print(f"{'C3+':<12s} {'CIV 1548+1551':<20s} {f_civ_c:12.3e} {Cppp:12.3e} {12+np.log10(Cppp):10.3f}")

# N2+/H+ from NIII] 1749+1752
f_niii = 0.0
for name in ["NIII_1749", "NIII_1752"]:
    f, e = get_flux(result_uv, name)
    if np.isfinite(f) and f > 0:
        f_niii += f
Npp = np.nan
if f_niii > 0:
    f_niii_c, _ = dust_correct(f_niii, 0, 1750.0, Av)
    eps_niii = N3_atom.getEmissivity(Te_high, ne, wave=1749) + N3_atom.getEmissivity(Te_high, ne, wave=1752)
    Npp = (f_niii_c / f_Hb_c) * (j_Hb / eps_niii)
    print(f"{'N2+':<12s} {'NIII] 1749+1752':<20s} {f_niii_c:12.3e} {Npp:12.3e} {12+np.log10(Npp):10.3f}")

# N3+/H+ from NIV] 1483+1486
f_niv = 0.0
for name in ["NIV_1483", "NIV_1486"]:
    f, e = get_flux(result_uv, name)
    if np.isfinite(f) and f > 0:
        f_niv += f
Nppp = np.nan
if f_niv > 0:
    f_niv_c, _ = dust_correct(f_niv, 0, 1486.5, Av)
    eps_niv = N4_atom.getEmissivity(Te_high, ne, wave=1487)
    if name == "NIV_1483":
        eps_niv += N4_atom.getEmissivity(Te_high, ne, wave=1483)
    Nppp = (f_niv_c / f_Hb_c) * (j_Hb / eps_niv)
    print(f"{'N3+':<12s} {'NIV] 1483+1486':<20s} {f_niv_c:12.3e} {Nppp:12.3e} {12+np.log10(Nppp):10.3f}")

# N4+/H+ from NV 1239+1243 (manual emissivity)
f_nv = 0.0
for name in ["NV_1", "NV_2"]:
    f, e = get_flux(result_uv, name)
    if np.isfinite(f) and f > 0:
        f_nv += f
Npppp = np.nan
if f_nv > 0:
    f_nv_c, _ = dust_correct(f_nv, 0, 1240.0, Av)
    eps_nv = _nv_emissivity(Te_high, ne, 1239) + _nv_emissivity(Te_high, ne, 1243)
    if eps_nv > 0:
        Npppp = (f_nv_c / f_Hb_c) * (j_Hb / eps_nv)
        print(f"{'N4+':<12s} {'NV 1239+1243':<20s} {f_nv_c:12.3e} {Npppp:12.3e} {12+np.log10(Npppp):10.3f}")

## 7. C/O and N/O ratios

In [ ]:
# C/O = (C2+ + C3+) / O2+
C_total = sum(x for x in [Cpp, Cppp] if np.isfinite(x))
if C_total > 0 and Opp > 0:
    CO = C_total / Opp
    print(f"C/O  = (C2+ + C3+) / O2+ = {CO:.4f}")
    print(f"log(C/O) = {np.log10(CO):.3f}")
    C_ions = {"C2+": Cpp, "C3+": Cppp}
    for ion, val in C_ions.items():
        if np.isfinite(val):
            print(f"  {ion}/C = {val/C_total:.1%}")
else:
    print("C/O not available")

print()

# N/O = (N2+ + N3+ + N4+) / O2+
N_total = sum(x for x in [Npp, Nppp, Npppp] if np.isfinite(x))
if N_total > 0 and Opp > 0:
    NO_uv = N_total / Opp
    print(f"N/O  = (N2+ + N3+ + N4+) / O2+ = {NO_uv:.4f}")
    print(f"log(N/O) = {np.log10(NO_uv):.3f}")
    N_ions = {"N2+": Npp, "N3+": Nppp, "N4+": Npppp}
    for ion, val in N_ions.items():
        if np.isfinite(val):
            print(f"  {ion}/N = {val/N_total:.1%}")
else:
    print("UV N/O not available")

## 8. Monte Carlo error propagation

Pre-compute emissivity grids as a function of Te, then use vectorised
`np.interp` — same approach as notebook 08.

In [ ]:
n_mc = 10000
rng = np.random.default_rng(42)

# Pre-compute emissivity grids
print("Pre-computing emissivity grids ...")
Te_g = np.linspace(5000, 80000, 500)

# [OIII] auroral/nebular ratio R(Te)
R_g = np.array([
    O3.getEmissivity(T, ne, wave=4363) /
    (O3.getEmissivity(T, ne, wave=4959) + O3.getEmissivity(T, ne, wave=5007))
    for T in Te_g
])

# O2+
j_5007_g = np.array([O3.getEmissivity(T, ne, wave=5007) for T in Te_g])
j_Hb_g = np.array([get_j_Hb(T, ne) for T in Te_g])

# C2+ (CIII])
j_ciii_g = np.array([
    C3.getEmissivity(T, ne, wave=1907) + C3.getEmissivity(T, ne, wave=1909)
    for T in Te_g
])

# C3+ (CIV)
j_civ_g = np.array([
    C4.getEmissivity(T, ne, wave=1548) + C4.getEmissivity(T, ne, wave=1551)
    for T in Te_g
])

# N2+ (NIII])
j_niii_g = np.array([
    N3_atom.getEmissivity(T, ne, wave=1749) + N3_atom.getEmissivity(T, ne, wave=1752)
    for T in Te_g
])

# N3+ (NIV])
j_niv_g = np.array([N4_atom.getEmissivity(T, ne, wave=1487) for T in Te_g])

# N4+ (NV) — manual
j_nv_g = np.array([
    _nv_emissivity(T, ne, 1239) + _nv_emissivity(T, ne, 1243)
    for T in Te_g
])

print("Done.")

In [ ]:
# Collect all fluxes and errors
flux_names = {
    "OIII_4363": (result_opt, 4364.44),
    "OIII_4959": (result_opt, 4961.68),
    "OIII_5007": (result_opt, 5009.64),
    "HBETA":     (result_opt, 4864.04),
    "Ha":        (result_opt, REST_LINES_A["Ha"]),
    "CIII]_1907": (result_uv, 1906.68),
    "CIII]":      (result_uv, 1908.73),
    "CIV_1":      (result_uv, 1548.19),
    "CIV_2":      (result_uv, 1550.77),
    "NIII_1749":  (result_uv, 1748.65),
    "NIII_1752":  (result_uv, 1752.16),
    "NIV_1483":   (result_uv, 1483.32),
    "NIV_1486":   (result_uv, 1486.50),
    "NV_1":       (result_uv, 1238.82),
    "NV_2":       (result_uv, 1242.80),
}

# Draw MC samples
mc_fluxes = {}
for name, (res, rwave) in flux_names.items():
    f, e = get_flux(res, name)
    if np.isfinite(f) and np.isfinite(e) and e > 0:
        mc_fluxes[name] = rng.normal(f, e, size=n_mc)
    else:
        mc_fluxes[name] = np.full(n_mc, np.nan)

# Dust correction (vectorised) — derive Av per sample from Ha/Hb
k_Ha_1 = salim_attenuation(np.array([REST_LINES_A["Ha"]]), 1.0)[0]
k_Hb_1 = salim_attenuation(np.array([4864.04]), 1.0)[0]
dk = k_Ha_1 - k_Hb_1

bal = mc_fluxes["Ha"] / mc_fluxes["HBETA"]
with np.errstate(invalid="ignore", divide="ignore"):
    Av_s = np.where(
        (bal > 0) & (bal > 2.86),
        -2.5 * np.log10(2.86 / bal) / (0.4 * dk),
        0.0,
    )
Av_s = np.clip(Av_s, 0, None)

# k(lambda)/A_V for each rest wavelength
kv = {}
for name, (_, rwave) in flux_names.items():
    kv[name] = salim_attenuation(np.array([rwave]), 1.0)[0]

def dc(name):
    return mc_fluxes[name] * 10.0 ** (0.4 * kv[name] * Av_s)

# Dust-corrected MC fluxes
c_Hb   = dc("HBETA")
c_4363 = dc("OIII_4363")
c_4959 = dc("OIII_4959")
c_5007 = dc("OIII_5007")
c_ciii = dc("CIII]_1907") + dc("CIII]")
c_civ  = dc("CIV_1") + dc("CIV_2")
c_niii = dc("NIII_1749") + dc("NIII_1752")
c_niv  = dc("NIV_1483") + dc("NIV_1486")
c_nv   = dc("NV_1") + dc("NV_2")

# Te from [OIII] ratio
with np.errstate(invalid="ignore", divide="ignore"):
    ratio_s = c_4363 / (c_4959 + c_5007)
Te_s = np.interp(ratio_s, R_g, Te_g)
Te_s[~np.isfinite(ratio_s) | (ratio_s < R_g[0]) | (ratio_s > R_g[-1])] = np.nan

# Interpolate emissivities
bad = ~np.isfinite(Te_s)
def interp_g(grid):
    arr = np.interp(Te_s, Te_g, grid)
    arr[bad] = np.nan
    return arr

ej_5007 = interp_g(j_5007_g)
ej_Hb   = interp_g(j_Hb_g)
ej_ciii = interp_g(j_ciii_g)
ej_civ  = interp_g(j_civ_g)
ej_niii = interp_g(j_niii_g)
ej_niv  = interp_g(j_niv_g)
ej_nv   = interp_g(j_nv_g)

# Ionic abundances (vectorised)
with np.errstate(invalid="ignore", divide="ignore"):
    Opp_s   = (c_5007 / c_Hb) * (ej_Hb / ej_5007)
    Cpp_s   = np.where(c_ciii > 0, (c_ciii / c_Hb) * (ej_Hb / ej_ciii), np.nan)
    Cppp_s  = np.where(c_civ > 0, (c_civ / c_Hb) * (ej_Hb / ej_civ), np.nan)
    Npp_s   = np.where(c_niii > 0, (c_niii / c_Hb) * (ej_Hb / ej_niii), np.nan)
    Nppp_s  = np.where(c_niv > 0, (c_niv / c_Hb) * (ej_Hb / ej_niv), np.nan)
    Npppp_s = np.where((c_nv > 0) & (ej_nv > 0), (c_nv / c_Hb) * (ej_Hb / ej_nv), np.nan)

# C/O and N/O
def pos_or_zero(arr):
    return np.where(np.isfinite(arr) & (arr > 0), arr, 0.0)

C_s = pos_or_zero(Cpp_s) + pos_or_zero(Cppp_s)
N_s = pos_or_zero(Npp_s) + pos_or_zero(Nppp_s) + pos_or_zero(Npppp_s)
O_s = pos_or_zero(Opp_s)

with np.errstate(invalid="ignore", divide="ignore"):
    OH_s = np.where(O_s > 0, 12 + np.log10(O_s), np.nan)
    CO_s = np.where((C_s > 0) & (O_s > 0), np.log10(C_s / O_s), np.nan)
    NO_s = np.where((N_s > 0) & (O_s > 0), np.log10(N_s / O_s), np.nan)

# Report
good = np.isfinite(Te_s) & np.isfinite(OH_s)
print(f"Valid MC samples: {good.sum()}/{n_mc} ({good.sum()/n_mc*100:.0f}%)\n")

def report(arr, label, fmt=".3f"):
    v = arr[np.isfinite(arr)]
    if len(v) == 0:
        print(f"  {label}: no valid samples")
        return
    lo, med, hi = np.percentile(v, [16, 50, 84])
    print(f"  {label:20s} = {med:{fmt}}   +{hi-med:{fmt}} / -{med-lo:{fmt}}")

report(Te_s, "Te [K]", fmt=".0f")
report(OH_s, "12+log(O/H)")
report(CO_s, "log(C/O)")
report(NO_s, "log(N/O) [UV]")

## 9. Summary plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

quantities = [
    (Te_s, r"$T_e\,(\mathrm{O}^{2+})$ [K]", ".0f", (8000, 60000)),
    (OH_s, r"12+log(O/H)", ".3f", (6.5, 9.0)),
    (CO_s, r"log(C/O)", ".3f", (-2.0, 0.5)),
    (NO_s, r"log(N/O) [UV]", ".3f", (-2.5, 0.0)),
]

for ax, (arr, label, fmt, (xlo, xhi)) in zip(axes.flat, quantities):
    v = arr[np.isfinite(arr)]
    v = v[(v >= xlo) & (v <= xhi)]
    if len(v) == 0:
        ax.text(0.5, 0.5, "No valid samples", transform=ax.transAxes, ha="center")
        continue
    lo, med, hi = np.percentile(v, [16, 50, 84])

    ax.hist(v, bins=60, color="steelblue", alpha=0.7, density=True,
            range=(xlo, xhi))
    ax.axvline(med, color="k", lw=1.5,
               label=f"Median: {med:{fmt}}")
    ax.axvspan(lo, hi, alpha=0.12, color="steelblue", label="16-84 pct")

    ax.set_xlim(xlo, xhi)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8, loc="upper right")

fig.suptitle(f"UV Abundance MC Error Propagation  (n = {n_mc:,})", fontsize=13, y=1.01)
plt.tight_layout()